<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-11/AI_Modul_11.8_Praktikum_Motion_Detection.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 11.8: Praktikum Motion Detection
**Mata Kuliah:** Visi Komputer dan Pengolahan Citra Digital  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mengimplementasikan deteksi gerak sederhana menggunakan diferensiasi bingkai absolut (`cv2.absdiff`).
2. Menerapkan pemodelan latar belakang statistik adaptif `cv2.createBackgroundSubtractorMOG2`.
3. Melakukan penyaringan bayangan bergerak (*shadow suppression*) dan pembersihan morfologi.
4. Memvisualisasikan bounding box objek bergerak pada sekuens simulasi pos pemantauan kebun.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print(f'OpenCV Version: {cv2.__version__}')


## 1. Pembangkitan Sekuens Video Truk Buah Sawit Bergerak
Mensimulasikan sekuens video pemantauan jalan kebun dengan truk buah yang melintas beserta bayangan tanah.

In [ ]:
H, W = 350, 550
sequence = []

for step in range(35):
    bg = np.full((H, W, 3), 110, dtype=np.uint8) # Jalan tanah perkebunan
    # Objek Truk (x bergerak dari 40 ke 420)
    x_pos = int(40 + step * 11)
    y_pos = 150
    # Gambar truk buah
    cv2.rectangle(bg, (x_pos, y_pos), (x_pos + 110, y_pos + 60), (25, 95, 215), -1)
    # Gambar bayangan di tanah (lebih gelap)
    cv2.rectangle(bg, (x_pos + 10, y_pos + 60), (x_pos + 120, y_pos + 72), (70, 70, 70), -1)
    sequence.append(bg)

print(f'Jumlah Frame Sekuens Video Sintetis: {len(sequence)}')


## 2. Deteksi Gerak: Frame Differencing vs Background Subtraction MOG2
Membandingkan hasil deteksi selisih frame sederhana dan model campuran Gaussian MOG2.

In [ ]:
# 1. Frame Differencing pada Frame #20 dan #21
gray_prev = cv2.cvtColor(sequence[19], cv2.COLOR_BGR2GRAY)
gray_curr = cv2.cvtColor(sequence[20], cv2.COLOR_BGR2GRAY)
diff = cv2.absdiff(gray_curr, gray_prev)
_, mask_diff = cv2.threshold(diff, 30, 255, cv2.THRESH_BINARY)

# 2. Background Subtraction MOG2 pada seluruh sekuens
mog2 = cv2.createBackgroundSubtractorMOG2(history=20, varThreshold=16, detectShadows=True)
mog2_masks = []
for f in sequence:
    fg = mog2.apply(f)
    mog2_masks.append(fg)

target_fg = mog2_masks[20]
# Nilai 255: Objek sejati; Nilai 127: Bayangan
_, mask_pure_obj = cv2.threshold(target_fg, 200, 255, cv2.THRESH_BINARY)

print(f'Piksel Aktif Frame Differencing : {cv2.countNonZero(mask_diff)}')
print(f'Piksel Aktif MOG2 (Tanpa Bayang): {cv2.countNonZero(mask_pure_obj)}')


## 3. Visualisasi Hasil Masker Deteksi Gerak dan Eliminasi Bayangan
Menampilkan perbandingan citra asli, Frame Differencing, masker mentah MOG2, dan masker bersih.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(cv2.cvtColor(sequence[20], cv2.COLOR_BGR2RGB))
axes[0].set_title('1. Frame Asli (Truk & Bayangan)', fontsize=10, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(mask_diff, cmap='gray')
axes[1].set_title('2. Frame Diff: Objek Berlubang', fontsize=10, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(target_fg, cmap='gray')
axes[2].set_title('3. MOG2 Mentah (Abu=Bayangan)', fontsize=10, fontweight='bold')
axes[2].axis('off')

axes[3].imshow(mask_pure_obj, cmap='gray')
axes[3].set_title('4. MOG2 Bersih (Bayangan Dieliminasi)', fontsize=10, fontweight='bold')
axes[3].axis('off')

plt.tight_layout()
plt.savefig('praktikum_motion_detection_11_8.png', dpi=150)
plt.show()
